# LNG features
Revised ALSI history, assumed available after two calendar days. The raw-file outlier replacement and its flag are retained; no extra outlier edits are made here.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get("TTF_PROJECT_ROOT", Path.cwd().parent))
RAW = ROOT / "1. Raw Data"
OUT = ROOT / "2. Data Processing"
lng = pd.read_csv(RAW / "lng_sendout.csv", parse_dates=["gasDayStart"])
lng = lng.sort_values("gasDayStart").reset_index(drop=True)

In [2]:
lng["lng7"] = lng["sendOut"].rolling(7, min_periods=7).sum() / 1000
lng["lng_dev"] = lng["lng7"] - lng["lng7"].shift(1).rolling(365, min_periods=365).mean()
lng = lng.drop(columns=["sendOut", "status"])
lng["lng_gas_day"] = lng["gasDayStart"]
lng["lng_available_date"] = lng["lng_gas_day"] + pd.Timedelta(days=2)
lng["lng_revised_proxy"] = True
lng["gasDayStart"] = lng["lng_available_date"]
lng.to_csv(OUT / "lng_processed.csv", index=False)